# 34장 · LangGraph의 선택적 도구 호출과 반복

검색을 도구로 둔 그래프는 질문에 따라 검색을 건너뛰거나 다시 요청할 수 있습니다. 고정 검색 경로와 실행 횟수가 달라지는 이유를 살펴봅니다.

같은 질문을 필수 검색 노드와 선택형 검색 도구에 보냅니다. 검색 횟수와 반환 문서를 함께 기록해 ‘검색 생략’과 ‘검색했으나 결과 없음’을 구분합니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 langgraph 1.2.16입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 조건부 간선으로 검색 생략

이 그래프는 질문에 대출·휴관·반납 중 한 단어라도 있는지 검사해 경로를 고릅니다. model에서 tool로 간 경우에만 rounds가 늘고 다시 model로 돌아옵니다. 실습의 한도는 검색 한 차례입니다. 생성 모델의 실제 도구 선택 성공률을 재는 예제는 아닙니다.


### 셀 01 · 조건부 도구 호출 그래프

decide에서 finish를 돌려주는 두 조건을 찾아보세요. 주차 요금 한 건을 넣습니다.


**실행 준비** · 30장의 문서 세 건과 retrieve를 불러옵니다. 아래 준비 셀을 한 번 실행한 뒤 이어지는 예제에서 이번 장의 변경을 확인하세요.

In [ ]:
records = [{'id': 'library-1',
  'text': '도서 대출 기간은 14일입니다.',
  'source': '가상 도서관 이용 안내',
  'page': 1},
 {'id': 'library-2',
  'text': '도서관은 월요일에 휴관합니다.',
  'source': '가상 도서관 이용 안내',
  'page': 2},
 {'id': 'library-3',
  'text': '무인 반납함은 24시간 이용할 수 있습니다.',
  'source': '가상 도서관 이용 안내',
  'page': 3}]
terms = ("대출", "휴관", "반납")

def retrieve(question, k=1):
    query = [int(term in question) for term in terms]
    if not any(query):
        return []
    def distance(row):
        vector = [int(term in row["text"]) for term in terms]
        return sum((a-b)**2 for a, b in zip(query, vector)), row["id"]
    return sorted(records, key=distance)[:k]


In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
class Loop(TypedDict):
    question: str
    rounds: int
    documents: list[dict]

def decide(state):
    if state["rounds"] >= 1:
        return "finish"
    if any(term in state["question"] for term in terms):
        return "search"
    return "finish"

def run_tool(state):
    return {"documents": retrieve(state["question"], 1), "rounds": state["rounds"] + 1}

builder = StateGraph(Loop)
builder.add_node("model", lambda state: {})
builder.add_node("tool", run_tool)
builder.add_edge(START, "model")
builder.add_conditional_edges("model", decide, {"search": "tool", "finish": END})
builder.add_edge("tool", "model")
agent = builder.compile()

request = {'question':'주차 요금은 얼마인가요?', 'rounds':0, 'documents':[]}
print(agent.invoke(request)['rounds'])

0


0은 검색을 실행하지 않았다는 기록입니다. 검색을 했지만 결과가 빈 경우도 문서 목록은 []이므로 목록만 봐서는 두 상황을 구별하기 어렵습니다.


#### 검색을 건너뛴 경로를 읽는 법

<mark>이 셀의 model 노드는 빈 갱신을 반환하고 decide가 질문과 rounds로 다음 노드를 정합니다.</mark> ‘주차 요금’에는 지정한 검색어가 없어 rounds가 0인 채 끝납니다. 검색어가 있으면 tool이 한 번 실행되고 model로 돌아옵니다. 두 번째 판단은 횟수 제한에 걸립니다.

선수 개념 · 27장 LangGraph 조건부 간선과 재검색

#### decide가 검색 또는 종료를 고르는 경로

```mermaid
flowchart TD
s["START"] --> m["model"]
m --> d{"rounds가 1 미만이고 검색어가 있는가?"}
d -->|search| t["tool · documents와 rounds 갱신"] --> m
d -->|finish| e["END"]
```

model이라는 노드 이름을 사용하지만 이 예제는 로컬 조건 함수로 분기합니다. tool이 rounds를 올리고 돌아온 뒤에는 검색어가 있어도 종료합니다.

### decide()가 반환하는 두 경로

<p>model 노드는 빈 변경분을 반환합니다. model 뒤의 조건 함수 decide가 rounds와 질문을 읽어 search 또는 finish를 반환합니다. 등록한 대응표에 따라 search는 tool로, finish는 END로 이어집니다.</p>


### 반복을 결정하는 상태 세 칸

<p>question은 어떤 자료가 필요한지 판단할 입력이고, documents는 방금 찾은 문서입니다. rounds는 tool을 실행할 때마다 1 증가합니다. 판단 함수 자체는 이 값을 변경하지 않습니다.</p>


### 셀 02 · 필수 검색 노드의 실행 경로

이번에는 분기 없이 검색을 거치는 실제 그래프입니다. 한 요청의 검색 횟수와 목록을 함께 읽으세요.


In [2]:
fixed = StateGraph(Loop)
fixed.add_node("search", run_tool)
fixed.add_edge(START, "search")
fixed.add_edge("search", END)
mandatory = fixed.compile()
fixed_result = mandatory.invoke(request)
print(fixed_result["rounds"], fixed_result["documents"])

1 []


1과 빈 목록이 나옵니다. 같은 질문에서 선택 검색은 위 셀처럼 0회, 필수 검색은 1회 호출 뒤 빈 결과였습니다.

검색 횟수 1과 빈 목록 []을 함께 읽으세요. 이 경로는 검색 단계를 반드시 지나지만, 자료에 주차 안내가 새로 생기지는 않습니다. 호출 여부를 점검하는 테스트와 근거가 있는지를 점검하는 테스트가 각각 필요한 사례입니다.


## 도구 호출 후 재분기와 종료 한도

일단 선택한 도구의 결과가 상태에 들어오면 다시 판단합니다. 완료한 검색을 세지 않거나 종료 조건보다 검색 조건을 먼저 검사하면 같은 요청을 되풀이할 수 있습니다.


### 도구가 돌려준 뒤 같은 model에서 종료 경로를 고릅니다

model → tool → model로 한 바퀴 도는 동안 rounds가 0에서 1로 바뀝니다. 두 번째 decide가 finish를 반환하므로 tool에 다시 들어가지 않습니다.

1. START · question = 대출 기간은 며칠인가요?
2. model · decide의 반환 = search
3. tool · documents[0].id = library-1
4. model · decide의 반환 = finish
5. END · 출력 = 1 library-1


### 셀 03 · 도구 요청과 상태 추적

같은 초기 상태에 대출 질문을 넣어 결과와 검색 횟수를 확인합니다.

이 그래프에서 대출 질문은 model → tool → model → END를 지납니다. model 노드는 실제 언어 모델 호출 없이 조건 분기에 제어를 넘깁니다.


In [3]:
result = agent.invoke({**request, 'question':'대출 기간은 며칠인가요?'})
print(result['rounds'], result['documents'][0]['id'])

1 library-1


1회 검색에서 library-1을 찾았습니다. tool 뒤에 model로 돌아왔지만 rounds가 1이어서 두 번째 검색을 요청하지 않습니다.


### 상한 검사를 먼저 하는 이유

<p>도구가 호출되면 rounds가 1이 됩니다. 질문에는 여전히 ‘대출’이 들어 있으므로 decide가 단어부터 검사하면 같은 검색을 다시 요청할 수 있습니다. 예제는 rounds부터 검사해 두 번째 방문에서 종료합니다.</p>


### 검색 생략이 적절했는지 판단하기

<p>규칙은 실제 의도 대신 질문에 대출·휴관·반납이라는 낱말이 있는지만 봅니다. 예를 들어 “책을 빌릴 수 있는 기간?”은 대출 질문이지만 이 조건에서는 검색을 생략합니다. rounds=0만 보고 올바른 응답이라고 결론짓기 어렵습니다.</p>


### 과제 04 · 반복 상한의 종료 분기

대출 질문이라도 이미 한 번 검색했다면 decide가 무엇을 반환하는지 확인하세요.


In [4]:
# 결과 상태 result를 판단 함수에 전달하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>print(decide(result))</code></pre>
<p>실행 결과</p><pre><code>finish</code></pre>
<p>finish가 돌아옵니다. 필수 검색이 서비스 규칙이면 누락을 막기 위해 고정 노드를 두고, 선택 검색을 쓴다면 생략과 상한 종료를 따로 기록하세요.</p>
</details>


### 과제 05 · 동의어 질문에서 검색 생략 진단

필수 그래프와 선택 그래프에 “책을 빌릴 수 있는 기간은?”을 각각 넣으세요. 검색 횟수와 문서 목록을 비교하고, 선택 규칙만 고치면 충분한지 설명하세요.


In [ ]:
request2 = {**request, 'question': '책을 빌릴 수 있는 기간은?'}
# 두 그래프를 실행하고 rounds와 documents를 확인하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>request2 = {**request, &#x27;question&#x27;: &#x27;책을 빌릴 수 있는 기간은?&#x27;}
optional = agent.invoke(request2)
required = mandatory.invoke(request2)
print(optional[&#x27;rounds&#x27;], optional[&#x27;documents&#x27;])
print(required[&#x27;rounds&#x27;], required[&#x27;documents&#x27;])</code></pre>
<p>실행 결과</p><pre><code>0 []
1 []</code></pre>
<p>선택 그래프는 검색을 생략했고 필수 그래프는 한 번 검색했습니다. 둘 다 ‘빌릴’을 ‘대출’로 연결하지 못했습니다. 선택 분기만 바꾸면 필수 그래프의 출력처럼 검색은 실행되지만 문서는 비어 있습니다. 검색 함수의 질의 표현도 점검해야 합니다.</p>
</details>


## 돌아보기

검색이 반드시 필요한 팀 서비스인지 먼저 합의하세요. 한 번 더 판단할 때의 비용과 검색 누락을 확인할 사례를 함께 정해야 구조를 고를 수 있습니다.


## 도구 결과를 모델 노드로 되돌려 루프 완성하기

33장 그래프는 도구를 한 번 실행하고 끝났습니다. 이번 그래프는 도구 결과가 상태에 붙으면 모델 노드로 돌아갑니다. 같은 경로를 여러 번 관찰할 수 있도록 모델의 선택과 최종 문장은 수업용 함수로 고정하고, 도구와 조건부 간선은 실제 LangGraph에서 실행합니다.

### 셀 07 · 모델 노드가 서로 다른 두 메시지를 내는 자리

처음에는 사용자 질문을 읽고, 도구가 실행된 뒤에는 ToolMessage의 결과를 읽습니다. 테스트용 모델 함수가 대출 질문에 보낸 첫 메시지에서 호출 이름을 확인하세요.

In [5]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

@tool
def find_notice(question: str) -> str:
    """가상 도서관의 대출·휴관·반납 안내를 한 건 찾습니다."""
    docs = retrieve(question, 1)
    return docs[0]["text"] if docs else "확인할 자료가 없습니다."

def scripted_decision(state):
    last = state["messages"][-1]
    if isinstance(last, ToolMessage):
        reply = AIMessage(content=last.content)
    elif any(term in last.content for term in terms):
        reply = AIMessage(content="", tool_calls=[{
            "id": "lookup-1", "name": find_notice.name,
            "args": {"question": last.content}, "type": "tool_call",
        }])
    else:
        reply = AIMessage(content="검색 요청이 없습니다.")
    return {"messages": [reply]}

loan_input = {"messages": [HumanMessage(content="대출 기간은?")]}
first_choice = scripted_decision(loan_input)["messages"][0]
print(first_choice.tool_calls[0]["name"], first_choice.tool_calls[0]["id"])

find_notice lookup-1


find_notice와 lookup-1이 나옵니다. 아직 검색이 실행되지는 않았습니다. 두 번째로 이 함수를 방문할 때에는 마지막 메시지가 ToolMessage라서 도구 요청이 없는 최종 답변을 만듭니다.

### 모델 재호출과 종료 시점

```mermaid
flowchart TD
 begin["START"] --> m["model · 마지막 메시지 읽기"]
 m --> has{"도구 요청이 있는가?"}
 has -->|있음| t["tools · find_notice 실행"]
 t -->|ToolMessage 추가| m
 has -->|없음| finish["END · 마지막 답변"]
```

도구 호출을 작성한 model은 아직 최종 답변을 쓰지 않았습니다. tools가 돌려준 메시지를 읽은 다음 방문에서 호출이 비면 종료합니다.

### 셀 08 · 모델을 두 번 지나는 간선 연결하기

모델 노드에서 tools_condition을 확인하고 tools를 지나면 model로 돌아오게 합니다. 대출 질문 한 건에서 나온 노드 이름의 순서를 읽으세요.

In [6]:
def build_dialog(decision):
    g = StateGraph(MessagesState)
    g.add_node("model", decision)
    g.add_node("tools", ToolNode([find_notice]))
    g.add_edge(START, "model")
    g.add_conditional_edges("model", tools_condition, {"tools": "tools", "__end__": END})
    g.add_edge("tools", "model")
    return g.compile()

dialog = build_dialog(scripted_decision)
trace = list(dialog.stream(loan_input, stream_mode="updates"))
print([next(iter(update)) for update in trace])
print(trace[-1]["model"]["messages"][0].content)

['model', 'tools', 'model']
도서 대출 기간은 14일입니다.


model, tools, model이 차례로 나옵니다. 두 번째 model에서 최종 문장을 썼기 때문에 END에서 이용자에게 전할 대출 기간 안내가 남습니다. 요청이 없는 인사말은 첫 번째 model에서 끝납니다.

### 과제 09 · 별칭을 검색 도구의 인자로 바꾸기

“책을 빌리는 기간은?”에는 대출이라는 낱말이 없어 기존 판단 함수가 검색을 생략합니다. 이 표현을 만난 첫 차례에만 검색용 질문을 “대출 기간은?”으로 바꾸는 노드를 만드세요. 원래의 사용자 메시지는 보존하고, 실행 뒤 사용자 질문·도구에 보낸 질문·최종 문장을 비교하세요.

In [7]:
alias_input = {"messages": [HumanMessage(content="책을 빌리는 기간은?")]}
# HumanMessage일 때만 검색용 질문을 바꾸고 기존 scripted_decision을 재사용하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>alias_input = {&quot;messages&quot;: [HumanMessage(content=&quot;책을 빌리는 기간은?&quot;)]}
def alias_decision(state):
    latest = state[&quot;messages&quot;][-1]
    if isinstance(latest, HumanMessage) and &quot;빌리&quot; in latest.content:
        lookup_state = {&quot;messages&quot;: [HumanMessage(content=&quot;대출 기간은?&quot;)]}
        return scripted_decision(lookup_state)
    return scripted_decision(state)

alias_graph = build_dialog(alias_decision)
history = alias_graph.invoke(alias_input)[&quot;messages&quot;]
print(history[0].content)
print(history[1].tool_calls[0][&quot;args&quot;][&quot;question&quot;])
print(history[-1].content)</code></pre>
<p>실행 결과</p><pre><code>책을 빌리는 기간은?
대출 기간은?
도서 대출 기간은 14일입니다.</code></pre>
<p>사용자가 물은 문장은 그대로 남고, 도구가 받은 질문만 대출 기간으로 바뀝니다. 마지막에는 14일 안내가 나옵니다. 검색을 하겠다는 결정과 검색기가 알아볼 질의를 만드는 작업이 모두 맞아야 이 경로가 완성됩니다.</p>
</details>

### 셀 10 · 선택 실습 · 실제 모델 노드로 같은 그래프 실행하기

33장 선택 실습에서 연결한 도구 호출 지원 채팅 모델을 이 노트북에도 준비한 뒤 live_model에 넣으세요. 호출 요청을 담은 AIMessage가 두 번 나오면 다음 차례에는 도구를 연결하지 않은 모델로 답변을 작성합니다. 실제 모델은 검색을 쓸지와 도구 인자를 다르게 선택할 수 있습니다.

In [8]:
live_model = None  # 준비한 채팅 모델 객체로 교체
if live_model is None:
    print("모델을 연결하면 이 셀에서 실제 선택 경로를 실행합니다.")
else:
    bound_model = live_model.bind_tools([find_notice])
    def live_decision(state):
        messages = state["messages"]
        rounds = sum(isinstance(m, AIMessage) and bool(m.tool_calls) for m in messages)
        current_model = live_model if rounds >= 2 else bound_model
        return {"messages": [current_model.invoke(messages)]}

    live_graph = build_dialog(live_decision)
    for update in live_graph.stream(loan_input, config={"recursion_limit": 12}, stream_mode="updates"):
        node, delta = next(iter(update.items()))
        last_message = delta["messages"][-1]
        print(node, last_message.type, last_message.content or "도구 요청")

모델을 연결하면 이 셀에서 실제 선택 경로를 실행합니다.


첫 model이 도구를 선택하면 tools 뒤에 model이 다시 나옵니다. 이때 실제 제공자의 호출량이 늘어나는 지점은 model 노드에서 채팅 모델을 부를 때입니다. find_notice는 로컬 자료를 읽습니다. 출력의 도구 요청 묶음 수와 도구 실행 횟수가 꼭 같지는 않습니다. 모델 한 차례가 여러 호출을 요청할 수 있기 때문입니다.

<mark>도구 뒤의 간선이 모델로 돌아가야 도구 결과를 읽은 답변이 생깁니다.</mark> 같은 model 노드를 두 번 지나지만 첫 방문의 출력은 도구 요청, 두 번째 방문의 출력은 최종 답변입니다.

평가할 때에는 답 문장과 함께 실제로 거친 노드와 검색 호출 수를 보관하세요.